# PDF Retrieval-Augmented Generation (RAG) Pipeline using LangChain and ChromaDB

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) pipeline by loading PDF documents, splitting them into chunks, generating embeddings, storing them in a vector database, and retrieving relevant documents for user queries.

In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204

In [97]:
from langchain_core.documents import Document

## Ingestion Pipeline

In [98]:
!pip install -q langchain langchain-community pypdf

In [4]:
# Data => Documents
import os
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_422/3544037227.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Document Loading

In [99]:
def load_all_pdfs():
    folder_path = "."
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [100]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [101]:
from collections import Counter

print(Counter(doc.metadata["source"] for doc in all_pdf_documents))

Counter({'./Research.pdf': 21, './attention.pdf': 11})


In [102]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

## Text Chunking

In [103]:
# chunks
!pip install langchain_text_splitters

In [104]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=1000, chunk_overlap=200):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [105]:
chunks = split_docs(all_pdf_documents)

In [106]:
len(chunks)

187

### Embedding Generation

In [107]:
from sentence_transformers import SentenceTransformer

In [108]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [16]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dimensions= 384


/tmp/ipykernel_422/1519971626.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


### Vector Database

In [109]:
import chromadb
import uuid

In [110]:
class VectorStoreManager:

    def __init__(
        self,
        persist_directory="data/vector_store",
        collection_name="pdf_documents1"
    ):

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):

        # Create directory
        os.makedirs(self.persist_directory, exist_ok=True)

        # Create persistent ChromaDB client
        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        # Create or load collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "Vector store collection for PDF embeddings in RAG"
            }
        )

        print(
            "Initialized vector store with collection:",
            self.collection_name
        )

        print(
            "Documents currently in collection:",
            self.collection.count()
        )

    def add_documents(self, documents, embeddings):

        # Check lengths
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents does not match number of embeddings"
            )

        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        # Prepare all documents
        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Unique ID
            doc_id = f"doc_{uuid.uuid4()}"

            ids.append(doc_id)

            # Metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            all_metadata.append(metadata)

            # Document text
            documents_content.append(doc.page_content)

            # Embedding
            embeddings_list.append(
                embedding.tolist()
            )

        # Add ALL documents ONCE
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print(
            "Total documents added:",
            len(documents_content)
        )

        print(
            "Documents in collection:",
            self.collection.count()
        )

In [111]:
vector_store = VectorStoreManager()

Initialized vector store with collection: pdf_documents1
Documents currently in collection: 119


In [112]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

embeddings shape: (187, 384)
Total documents added: 187
Documents in collection: 306


# Retrieval Pipeline

In [113]:
from sklearn.metrics.pairwise import cosine_similarity

In [114]:
class RAGRetriever:

    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):

        # Convert query into embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Search in ChromaDB
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved_docs = []

        # Check if results exist
        if results["documents"] and results["documents"][0]:

            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(
                zip(ids, metadatas, documents, distances)
            ):

                # ChromaDB cosine distance
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:

                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f"Retrieved {len(retrieved_docs)} documents")

        else:
            print("No documents found")

        return retrieved_docs

In [115]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [116]:
print(load_all_pdfs())

total pdfs: 2
total pages: 32
[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensem

In [117]:
from collections import Counter
print(len(chunks))
Counter(chunk.metadata["source"] for chunk in chunks)

187


Counter({'./attention.pdf': 43, './Research.pdf': 144})

# Integrate with LLMs - Groq

In [136]:
from google.colab import userdata

groq_api_key = userdata.get('grok')

In [137]:
from groq import Groq

client = Groq(api_key=groq_api_key)

In [29]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.7 MB/s eta 0:00:00


In [138]:
from langchain_groq import ChatGroq
from google.colab import userdata

llm = ChatGroq(
    groq_api_key=userdata.get("grok"),
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024
)

In [139]:
def generate_output(query, rag_retriever, llm):
    # Retrieve relevant documents
    retrieved_docs = rag_retriever.retrieve(
        query,
        top_k=5
    )

    # Handle no retrieved documents
    if not retrieved_docs:
        return "I could not find relevant information in the documents."

    # Combine retrieved document text
    context = "\n\n".join(
        doc["document"] for doc in retrieved_docs
    )

    # Create detailed prompt
    prompt = f"""
You are a knowledgeable AI assistant.

Your task is to answer the user's question ONLY using the information provided in the context below.

Context:
{context}

Question:
{query}

Instructions:
1. Use only the provided context.
2. Write a detailed answer of approximately 150–300 words.
3. Begin with a clear definition or introduction.
4. Explain the concept in simple and easy-to-understand language.
5. Include important points, features, advantages, working, or examples if they are present in the context.
6. Do NOT add information that is not available in the context.
7. Do NOT mention "according to the context" or "the provided document."
8. If the answer is not found in the context, reply exactly:
   "The answer is not available in the provided documents."

Detailed Answer:
"""

    # Generate response
    response = llm.invoke(prompt)

    # Return response text
    if hasattr(response, "content"):
        return response.content.strip()
    else:
        return str(response).strip()

## Testing

In [140]:
answer = generate_output("What are the components of the encoder layer?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 5 documents


In [141]:
print(answer)

The encoder layer is a crucial component of the Transformer model architecture. It is composed of a stack of N = 6 identical layers, with each layer consisting of two sub-layers. The first sub-layer is a multi-head self-attention mechanism, which allows the model to attend to different parts of the input sequence simultaneously. The second sub-layer is a simple, position-wise fully connected feed-forward network, which transforms the output of the self-attention mechanism.

These two sub-layers are connected in a specific way, with residual connections around each of them, followed by layer normalization. This means that the output of each sub-layer is computed as LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. This residual connection helps to facilitate the flow of information through the network and prevents the vanishing gradient problem.

All sub-layers in the model, including the embedding layers, produce outputs of dimension dmo

In [142]:
answer = generate_output("What is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 5 documents


In [143]:
print(answer)

RAG, or Retrieval-Augmented Generation, is a research paradigm that has evolved significantly, particularly with the integration of Large Language Models (LLMs). It involves the process of "Retrieval," "Generation," and "Augmentation" to form a cohesive framework. The RAG process has been categorized into three stages: Naive RAG, Advanced RAG, and Modular RAG. Naive RAG represents the earliest methodology, which has limitations that are addressed by Advanced RAG and Modular RAG. 

The RAG process is designed to equip readers and professionals with a detailed understanding of both large models and RAG. It aims to illuminate the evolution of retrieval augmentation techniques, assess the strengths and weaknesses of various approaches, and speculate on upcoming trends and innovations. The RAG research paradigm has undergone significant development, with a focus on evaluating RAG models through benchmark tests and tools. These instruments provide quantitative metrics to gauge RAG model perf

In [144]:
questions = [
    "What is the attention mechanism?",
    "How does self-attention work?",
    "What is multi-head attention?",
    "What is Retrieval-Augmented Generation (RAG)?",
    "What are the three stages of a Naive RAG pipeline?"
]

for i, question in enumerate(questions, 1):
    print("=" * 100)
    print(f"Question {i}: {question}\n")

    answer = generate_output(question, rag_retriever, llm)

    print("Answer:")
    print(answer)
    print("\n")

Question 1: What is the attention mechanism?



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 5 documents
Answer:
The attention mechanism is a function that maps a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. This mechanism allows the model to focus on specific parts of the input data that are relevant to the task at hand. In the case of self-attention, also known as intra-attention, it relates different positions of a single sequence to compute a representation of the sequence. This has been successfully used in various tasks such as reading comprehension, abstractive summarization, and learning task-independent sentence representations.

The attention function works by computing a weighted sum of the values based on the query and keys. This process allows the model to jointly attend to information from different representation subspaces at different positions. Multi-head attention is a variant of this mechanism, which enables the model to attend to information from different re

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 3 documents
Answer:
Self-attention is an attention mechanism that relates different positions of a single sequence to compute a representation of the sequence. It is also known as intra-attention. This mechanism has been successfully used in various tasks such as reading comprehension, abstractive summarization, textual entailment, and learning task-independent sentence representations. 

Self-attention works by allowing the model to inspect attention distributions and learn to perform different tasks. Individual attention heads can clearly learn to exhibit behavior related to the syntactic and semantic structure of sentences. This ability to learn and understand the structure of sentences makes self-attention a powerful tool for natural language processing tasks.

One of the benefits of self-attention is that it can yield more interpretable models. This means that the attention distributions from the models can be inspected and understood, providin

...

embeddings shape: (1, 384)
Retrieved 5 documents
Answer:
Multi-head attention is a mechanism that allows a model to jointly attend to information from different representation subspaces at different positions. This is achieved by using multiple attention heads in parallel, which enables the model to capture a wider range of contextual relationships. With a single attention head, averaging inhibits this ability, but multi-head attention overcomes this limitation.

In the multi-head attention mechanism, the queries, keys, and values are first projected using parameter matrices, and then the attention function is performed in parallel for each head. The output values from each head are then concatenated and projected again to produce the final output. This process allows the model to attend to different aspects of the input data simultaneously.

The Transformer model uses multi-head attention in three different ways, including "encoder-decoder attention" layers, where the queries come fro

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 5 documents
Answer:
Retrieval-Augmented Generation (RAG) is a concept that involves the integration of retrieval techniques with large language models (LLMs) to enhance their performance. The process of RAG mainly includes pre-training, fine-tuning, and inference stages. Initially, research on RAG focused on leveraging the powerful in-context learning abilities of LLMs, primarily concentrating on the inference stage. Over time, subsequent research has delved deeper, gradually integrating more with the fine-tuning of LLMs. 

The RAG process consists of three core components: "Retrieval", "Generation", and "Augmentation". These components intricately collaborate to form a cohesive and effective RAG framework. The "Retrieval" stage involves optimization methods, including indexing, query, and embedding optimization. The "Generation" stage concentrates on post-retrieval processes and LLM fine-tuning. The "Augmentation" stage analyzes the three augmentat

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
Retrieved 2 documents
Answer:
The RAG research paradigm is categorized into three stages: Naive RAG, Advanced RAG, and Modular RAG. However, the context does not explicitly outline the three stages of a Naive RAG pipeline. Instead, it introduces Naive RAG as the earliest methodology in the RAG research paradigm. 

Naive RAG represents the initial approach in the RAG process, but the context does not delve into the specifics of its pipeline stages. It does mention that the development of Advanced RAG and Modular RAG is a response to the limitations of Naive RAG, implying that Naive RAG has certain shortcomings. 

The context highlights the importance of understanding the RAG process and its evolution, including the integration of RAG within LLMs. It discusses the core stages of "Retrieval," "Generation," and "Augmentation" but does not associate these stages specifically with Naive RAG. 

Given the information available, it is not possible to provide a detaile